In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt
import os
import seaborn as sns
from sklearn.metrics import roc_curve, auc, confusion_matrix
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.preprocessing import label_binarize
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

Load the dataset;Check the number of samples amd classes in each spilt

In [ ]:
# Custom dataset class
class DermaMNISTDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

# Data preprocessing
data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])

# Load .npz file
data = np.load('dermamnist_64.npz')
x_train = data['train_images']
y_train = data['train_labels']
x_val = data['val_images']
y_val = data['val_labels']
x_test = data['test_images']
y_test = data['test_labels']

# Print dataset information
print("Dataset Information:")
print("-" * 50)
print(f"Training set:")
print(f"Images shape: {x_train.shape}")
print(f"Labels shape: {y_train.shape}")
print(f"Number of training samples: {len(x_train)}")
print("-" * 50)
print(f"Validation set:")
print(f"Images shape: {x_val.shape}")
print(f"Labels shape: {y_val.shape}")
print(f"Number of validation samples: {len(x_val)}")
print("-" * 50)
print(f"Test set:")
print(f"Images shape: {x_test.shape}")
print(f"Labels shape: {y_test.shape}")
print(f"Number of test samples: {len(x_test)}")
print("-" * 50)
print(f"Image size: {x_train.shape[1]}x{x_train.shape[2]}")
print(f"Number of channels: {x_train.shape[3] if len(x_train.shape) > 3 else 1}")
print(f"Number of unique labels: {len(np.unique(y_train))}")
print(f"Labels: {np.unique(y_train)}")

# Set batch size
BATCH_SIZE = 128

# Create dataset instances
train_dataset = DermaMNISTDataset(x_train, y_train, transform=data_transform)
val_dataset = DermaMNISTDataset(x_val, y_val, transform=data_transform)
test_dataset = DermaMNISTDataset(x_test, y_test, transform=data_transform)

# Create data loaders
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Examine class distribution for potential imbalance

In [ ]:
# Count the number of samples in each class
train_class_counts = np.bincount(y_train.flatten())
val_class_counts = np.bincount(y_val.flatten())
test_class_counts = np.bincount(y_test.flatten())

# Create a bar chart
plt.figure(figsize=(12, 6))
x = np.arange(len(train_class_counts))
width = 0.25

plt.bar(x - width, train_class_counts, width, label='Train')
plt.bar(x, val_class_counts, width, label='Validation')
plt.bar(x + width, test_class_counts, width, label='Test')

plt.xlabel('Class Label')
plt.ylabel('Number of Samples')
plt.title('Class Distribution in Train/Val/Test Sets')
plt.legend()
plt.xticks(x)
plt.show()

# Print the specific numbers and proportions
print("\
Class distribution in training set:")
for class_idx, count in enumerate(train_class_counts):
    percentage = (count / len(y_train)) * 100
    print(f"Class {class_idx}: {count} samples ({percentage:.2f}%)")

Visualization Images

In [ ]:
def show_images(dataloader, rows=8, cols=8):
    num_images = rows * cols

    # Get a batch of data
    images, labels = next(iter(dataloader))

    # Ensure the requested number of images does not exceed the batch size
    num_images = min(num_images, len(images))

    # Adjust rows and cols to fit the actual number of images
    rows = int(np.ceil(np.sqrt(num_images)))
    cols = int(np.ceil(num_images / rows))

    # Create subplot grid
    fig, axes = plt.subplots(rows, cols, figsize=(cols*2, rows*2))
    axes = axes.ravel()  # Flatten the 2D array into 1D

    for i in range(num_images):
        # Get the image and convert it to a numpy array
        img = images[i].numpy()

        # If it's a 3-channel image, transpose the dimensions
        if img.shape[0] == 3:
            img = np.transpose(img, (1, 2, 0))

        # Denormalize the image
        img = img * 0.5 + 0.5

        # Display the image
        axes[i].imshow(img, cmap='gray' if img.shape[-1] == 1 else None)
        axes[i].axis('off')
        axes[i].set_title(f'Label: {labels[i].item()}')

    # Hide extra subplots
    for i in range(num_images, len(axes)):
        axes[i].axis('off')
        axes[i].set_visible(False)

    plt.tight_layout()
    plt.show()

# Call the function to display images from the training set
# You can customize the number of rows and columns
show_images(train_loader, rows=10, cols=10)

ResNetCNN-18

define the ResNet18_CNN

In [ ]:
from torchvision.models import resnet18

n_classes = 7

model = resnet18(num_classes=n_classes).cuda()

Training Phase

In [ ]:
# Set training parameters
NUM_EPOCHS = 100
BATCH_SIZE = 128
lr = 0.001
GRAD_CLIP = 1.0
PATIENCE = 20
early_stopping_counter = 0
save_path='CNN'

# Create save directory if it doesn't exist
os.makedirs(save_path, exist_ok=True)

# Optimizer and loss function settings
optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss().cuda()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
scaler = GradScaler()

# Initialize best validation metrics
best_val_acc = 0.0
best_val_loss = float('inf')

def train_epoch(model, train_loader):
    model.train()
    train_loss = 0.0
    correct = 0
    total = 0

    for inputs, targets in train_loader:
        inputs, targets = inputs.cuda(), targets.cuda()

        optimizer.zero_grad(set_to_none=True)

        with autocast():
            outputs = model(inputs)
            targets = targets.squeeze().long()
            loss = criterion(outputs, targets)
            _, predictions = torch.max(outputs, 1)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        total += targets.size(0)
        correct += (predictions == targets).sum().item()

    return train_loss / len(train_loader), 100 * correct / total

def validate(model, val_loader):
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.cuda(), targets.cuda()

            with autocast():
                outputs = model(inputs)
                targets = targets.squeeze().long()
                loss = criterion(outputs, targets)
                _, predictions = torch.max(outputs, 1)

            val_loss += loss.item()
            total += targets.size(0)
            correct += (predictions == targets).sum().item()

    return val_loss / len(val_loader), 100 * correct / total

# Training loop
for epoch in range(NUM_EPOCHS):
    # Train and validate
    train_loss, train_acc = train_epoch(model, train_loader)
    val_loss, val_acc = validate(model, val_loader)

    # Update learning rate
    scheduler.step()

    # Print progress
    print(f'Epoch {epoch+1}/{NUM_EPOCHS} | '
          f'Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | '
          f'Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}% | '
          f'LR: {scheduler.get_last_lr()[0]:.6f}')

    # Model saving and early stopping
    if val_loss < best_val_loss or val_acc > best_val_acc:
        if val_acc > best_val_acc:
            best_val_acc = val_acc
        if val_loss < best_val_loss:
            best_val_loss = val_loss

        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_acc': val_acc,
            'val_loss': val_loss,
            'scaler': scaler.state_dict(),
        }, os.path.join(save_path, 'CNN_best_model.pth'))

        print(f'Model saved (Val Acc: {val_acc:.2f}%, Val Loss: {val_loss:.4f})')
        early_stopping_counter = 0
    else:
        early_stopping_counter += 1

    if early_stopping_counter >= PATIENCE:
        print(f'Early stopping triggered after {epoch+1} epochs')
        break

# Load the best model
print(f'Loading best model (Val Acc: {best_val_acc:.2f}%)')
checkpoint = torch.load(os.path.join(save_path, 'CNN_best_model.pth'))
model.load_state_dict(checkpoint['model_state_dict'])

In [ ]:
# Load the best model
save_path = 'CNN'
checkpoint = torch.load(os.path.join(save_path, 'CNN_best_model.pth'))
model.load_state_dict(checkpoint['model_state_dict'])

# Test set evaluation
model.eval()
y_true = []
y_score = []
attention_maps = [] # This variable is declared but not used.  Consider removing.
original_images = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.cuda(), targets.cuda()

        # Get model output and attention map (attention map not currently used)
        outputs = model(inputs)
        outputs = outputs.softmax(dim=-1)

        # Store original images
        original_images.extend(inputs.cpu().numpy())
        y_true.extend(targets.cpu().numpy())
        y_score.extend(outputs.cpu().numpy())

y_true = np.array(y_true)
y_score = np.array(y_score)
y_pred = np.argmax(y_score, axis=1)

# 1. Calculate metrics
accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')

print(f'Accuracy: {100*accuracy:.3f}%')
print(f'Precision: {100*precision:.3f}%')
print(f'Recall: {100*recall:.3f}%')
print(f'F1-score: {100*f1:.3f}%')

# 2. Plot ROC curves
plt.figure(figsize=(10, 8))
y_true_bin = label_binarize(y_true, classes=range(n_classes)) # n_classes needs to be defined earlier
fpr = dict()
tpr = dict()
roc_auc = dict()

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_score[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])
    plt.plot(fpr[i], tpr[i], label=f'Class {i} (AUC = {roc_auc[i]:.2f})')

plt.plot([0, 1], [0, 1], 'k--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend(loc="lower right")
plt.show()

# 3. Plot confusion matrix
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_true, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
sns.heatmap(cm_normalized, annot=True, fmt='.1f', cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Normalized Confusion Matrix (%)')
plt.show()

# 4. Visualize original images and attention maps using Grad-CAM
def visualize_gradcam(model, original_images, predictions, num_samples=5):
    # Determine target layer (usually the last convolutional layer)  - NEEDS ADJUSTMENT BASED ON YOUR MODEL
    target_layers = [model.layer4[-1]] #  This line needs to be adapted to your model's architecture.

    # Initialize GradCAM
    cam = GradCAM(model=model, target_layers=target_layers)

    fig, axes = plt.subplots(num_samples, 2, figsize=(10, 2*num_samples))
    for idx in range(num_samples):
        # Display original image
        orig_img = original_images[idx].transpose(1, 2, 0)
        # Normalize image for display
        orig_img = (orig_img - orig_img.min()) / (orig_img.max() - orig_img.min())
        axes[idx, 0].imshow(orig_img)
        axes[idx, 0].set_title(f'Original (Pred: {predictions[idx]})')
        axes[idx, 0].axis('off')

        # Get GradCAM
        input_tensor = torch.FloatTensor(original_images[idx:idx+1]).cuda()
        targets = [ClassifierOutputTarget(predictions[idx])]
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)

        # Overlay GradCAM on original image
        visualization = show_cam_on_image(orig_img, grayscale_cam[0], use_rgb=True)

        axes[idx, 1].imshow(visualization)
        axes[idx, 1].set_title('GradCAM Visualization')
        axes[idx, 1].axis('off')

    plt.tight_layout()
    plt.show()

# Call visualization function
visualize_gradcam(model, original_images, y_pred, num_samples=5)